# Vivaah — WAN 2.1 Text-to-Video (hero campaign film, 4 shots)

Generates the 4 hero shots for the homepage hero film **from text only** (no reference photos needed), on a **free Colab T4**.

This is the *text-to-video* companion to `wan_i2v_colab.ipynb` (which was image-to-video for the turntable product clips). Use THIS notebook for the hero film.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

**What it makes:** 4 shots × 2–4 takes each, 832×480 @16fps, saved to your Google Drive. We (OpenMontage) pick the best take per shot and do the grade / upscale / ping-pong-loop encode later — you do **not** need to choose takes or worry about resolution, trimming, or color.

**Order:** 1) GPU check → 2) install → 3) mount Drive → 4) config (already filled in) → 5) load model → 6) generate.

> ⚠️ **Timing on the free T4:** every shot uses the fast **1.3B** model — expect roughly **60–100 minutes total** for all 11 takes (~3–12 min per clip depending on length). The notebook **skips takes it already made**, so you can stop and resume across sessions.
>
> ⚠️ **The 14B model does NOT run on free Colab.** Its bf16 weights are ~28GB and CPU offload parks them in system RAM — free Colab has only ~12.7GB, so loading it kills the session before a single frame renders. 14B is an opt-in upgrade **only** on a high-RAM runtime (Colab Pro "High-RAM", or a local machine with ≥32GB RAM + a CUDA GPU). See section 5.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install dependencies (~2–3 min)

In [ ]:
!pip install -q git+https://github.com/huggingface/diffusers
!pip install -q transformers accelerate ftfy safetensors sentencepiece imageio imageio-ffmpeg bitsandbytes

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Config — already filled in for the 4 hero shots

You normally don't need to edit anything here. The 4 prompts, the shared negative prompt, and per-shot settings match `specs/WAN_COLAB_WORK_ORDER.md`.

- `model`: **all four shots default to `"1.3b"`** — the only size that runs on free Colab. Quality on the face shots (2 & 4) comes from **more takes** (3–4 each); we pick the best one downstream.
- `"14b"` is an **opt-in upgrade that will NOT run on free Colab** (needs a high-RAM runtime: Colab Pro High-RAM or a local ≥32GB-RAM GPU box). Only set a shot's `model` to `"14b"` if you're on such a runtime — the loader in section 5 checks your RAM and stops immediately with an explanation otherwise.
- `takes`: how many variations to generate (we pick the best one). 4 for the hero shot, 3 for the twirl, 2 for the rest. To re-run a shot, delete its mp4s in Drive and run cell 6 again.

In [ ]:
import os
# Must be set before torch touches CUDA: prevents the fragmentation
# ("reserved but unallocated") that OOMed the text-encoder load on T4.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

OUTPUT_ROOT = "/content/drive/MyDrive/vivaah/heroSection_v2/raw"  # shot folders are created under here

WIDTH, HEIGHT = 832, 480     # WAN 480P 16:9 (both divisible by 16). Upscaled to 1280x720 later.
FPS           = 16
FLOW_SHIFT    = 3.0          # 3.0 is correct for 480P (5.0 is for 720P)

# Shared negative prompt (from the brief + WAN-specific quality guards)
NEGATIVE_PROMPT = (
    "man, male, boy, groom, men, children, kid, text, watermark, logo, subtitles, caption, "
    "western wedding dress, white gown, church, cross, empty background, deserted room, "
    "deformed hands, extra fingers, mutated hands, mutated face, blurry face, two heads, "
    "fused fingers, flicker, strobing, jump cut, fast cut, camera shake, oversaturated, "
    "neon colors, cartoon, anime, CGI look, plastic skin, low quality, low resolution, "
    "jpeg artifacts, distorted, morphing, warping"
)

# model: "1.3b" everywhere by default — the only size that runs on FREE Colab.
# "14b" is opt-in and needs a high-RAM runtime (Colab Pro High-RAM / local >=32GB RAM);
# on free Colab the section-5 loader refuses it immediately instead of crashing the session.
SHOTS = {
    "shot1": {
        "model": "1.3b",
        "num_frames": 33,    # ~2.0s @16fps (target 1.5s, extra room to trim)
        "steps": 30,
        "guidance": 5.0,
        "takes": 2,
        "seeds": [42, 7, 123],
        "prompt": (
            "Extreme close-up of an Indian woman's mehendi-decorated hands wearing a red chooda and gold "
            "bangles, fingertips gliding slowly across dense gold zardozi embroidery on deep-red bridal silk. "
            "Warm candlelight travels along the raised metallic thread; sequins and beadwork catch tiny glints. "
            "Soft violet-dark background, shallow depth of field, gentle slow push-in. Rich reds and antique gold "
            "against near-black violet, warm tungsten glow, fine film grain, smooth stable slow motion. Editorial "
            "luxury fashion campaign, intimate and tactile, no face in frame, women only."
        ),
    },
    "shot2": {
        "model": "1.3b",     # face shot: quality via best-of-3 takes (14b only on high-RAM runtimes)
        "num_frames": 49,    # ~3.0s @16fps (target 2.0s)
        "steps": 30,
        "guidance": 5.5,
        "takes": 3,
        "seeds": [42, 7, 123],
        "prompt": (
            "Slow-motion medium-wide shot of a graceful Indian woman in her twenties wearing a flowing violet and "
            "gold side lehenga with a full flared skirt, twirling in a candlelit haveli courtyard at dusk. Her sheer "
            "dupatta lifts and floats on the air, the skirt hem flaring in a wide circle. Two or three women in "
            "colourful sarees stand softly out of focus behind her, gently clapping. Warm gold candle glow, deep "
            "violet shadows, bokeh lights, shallow depth of field, slow gentle camera orbit. Elegant, premium, "
            "cinematic, fine film grain, smooth motion, women only."
        ),
    },
    "shot3": {
        "model": "1.3b",
        "num_frames": 33,    # ~2.0s @16fps (target 1.5s)
        "steps": 30,
        "guidance": 5.5,
        "takes": 2,
        "seeds": [42, 7, 123],
        "prompt": (
            "Medium shot of an elegant Indian woman in a Banarasi silk saree with a broad gold zari border, "
            "descending a carved stone haveli staircase at dusk. Her gold jhumka earrings sway with each step; "
            "marigold and rose garlands drape the stone pillars beside her. Warm candlelight and hanging lamps, "
            "deep violet twilight, shallow depth of field, a slow graceful camera tilt down following her descent. "
            "Regal, unhurried, luxurious Indian bridal campaign, warm gold and violet palette, soft bokeh, fine "
            "film grain, smooth stable slow motion, women only."
        ),
    },
    "shot4": {
        "model": "1.3b",     # hero face shot: quality via best-of-4 takes (14b only on high-RAM runtimes)
        "num_frames": 81,    # ~5.0s @16fps (target 3.5s; extra tail room for the locked-off loop)
        "steps": 40,
        "guidance": 5.5,
        "takes": 4,
        "seeds": [42, 7, 123, 99],
        "prompt": (
            "Cinematic medium shot of an Indian bride in a deep-red and gold bridal lehenga with gold zardozi "
            "embroidery, mehendi hands, red chooda, maang tikka and jhumka earrings, facing camera with a calm "
            "confident expression. The camera pushes in very slowly for about one second, then stops and stays "
            "completely still. For the rest of the shot the camera is locked off and static: brass diya flames "
            "flicker, warm bokeh fairy lights shimmer, her dupatta and hair sway gently, she breathes and blinks "
            "softly. Three to five women in colourful sarees and lehengas stand softly out of focus behind her, "
            "swaying slightly in place. Warm gold candlelight, deep violet dusk, shallow depth of field, film grain. "
            "Nobody walks, no talking, nobody enters or leaves the frame. Elegant, premium, alive, women only."
        ),
    },
}

MODEL_IDS = {
    "1.3b": "Wan-AI/Wan2.1-T2V-1.3B-Diffusers",
    "14b":  "Wan-AI/Wan2.1-T2V-14B-Diffusers",
}

import os
for s in SHOTS:
    os.makedirs(os.path.join(OUTPUT_ROOT, s), exist_ok=True)
print("Config loaded. Outputs ->", OUTPUT_ROOT)
print("Shots:", {k: v["model"] for k, v in SHOTS.items()})

## 5. Model loader (downloads on first use)

The **1.3B** model loads straight onto the T4 — this is the free-tier path and the default for all shots.

> ⚠️ **14B will NOT run on free Colab.** Its bf16 weights are ~28GB (plus the umt5-xxl text encoder); `enable_model_cpu_offload()` keeps offloaded modules in **system RAM**, and free Colab pairs the T4 with only ~12.7GB RAM — loading is killed (or swap-thrashes the session to death) before a single frame renders. The loader below checks your runtime's RAM and **refuses `"14b"` instantly with this explanation** instead of letting you waste a 20-minute download on a doomed session. To actually use 14B: Colab Pro with a **High-RAM** runtime, or a local machine with ≥32GB RAM and a CUDA GPU.

**Update (RAM-crash fix):** if your session previously crashed with *"using all available RAM"*, that is fixed in this version — the big text encoder now loads 8-bit directly onto the GPU and never passes through system RAM. Just **Runtime → Run all** again (dependencies re-install after a crash; that's expected).


In [ ]:
import gc, torch, psutil
from transformers import UMT5EncoderModel
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler

_PIPES = {}
_MIN_RAM_GB_FOR_14B = 25

def _free_vram():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

def _load_text_encoder_gpu(model_id):
    # UMT5-XXL is ~11GB fp16 / ~22GB on disk. Load it quantized, shards streamed
    # straight to the GPU: 4-bit NF4 (~3GB, lowest load peak) -> 8-bit (~6GB) ->
    # fp16 (last resort). Free VRAM between attempts or the failed partial load
    # blocks the next one (that is what OOMed the first run's fallback).
    from transformers import BitsAndBytesConfig
    attempts = [
        ("4-bit NF4", BitsAndBytesConfig(load_in_4bit=True,
                                          bnb_4bit_quant_type="nf4",
                                          bnb_4bit_compute_dtype=torch.float16)),
        ("8-bit", BitsAndBytesConfig(load_in_8bit=True)),
        ("fp16", None),
    ]
    last_err = None
    for label, quant in attempts:
        try:
            kwargs = dict(subfolder="text_encoder", device_map={"": 0}, low_cpu_mem_usage=True)
            if quant is not None:
                kwargs["quantization_config"] = quant
            else:
                kwargs["torch_dtype"] = torch.float16
            enc = UMT5EncoderModel.from_pretrained(model_id, **kwargs)
            print(f"  text encoder loaded ({label})")
            return enc
        except Exception as e:
            print(f"  {label} load failed ({type(e).__name__}) — freeing VRAM and trying next option.")
            last_err = e
            _free_vram()
    raise last_err

def get_pipe(size):
    if size in _PIPES:
        return _PIPES[size]
    if size == "14b":
        ram_gb = psutil.virtual_memory().total / 1e9
        if ram_gb < _MIN_RAM_GB_FOR_14B:
            raise RuntimeError(
                f"14B needs a high-RAM runtime (>= {_MIN_RAM_GB_FOR_14B}GB system RAM, e.g. Colab Pro High-RAM); "
                f"this machine has {ram_gb:.1f}GB. On the free tier keep every shot on '1.3b'."
            )
    model_id = MODEL_IDS[size]
    print(f"Loading WAN 2.1 T2V {size} ({model_id}) ... first time downloads the model.")
    text_encoder = _load_text_encoder_gpu(model_id)
    vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)
    # fp16, not bf16: the T4 has no bf16 tensor cores — fp16 is native and ~2x faster.
    pipe = WanPipeline.from_pretrained(
        model_id, text_encoder=text_encoder, vae=vae, torch_dtype=torch.float16,
    )
    pipe.scheduler = UniPCMultistepScheduler(
        prediction_type="flow_prediction", use_flow_sigmas=True,
        num_train_timesteps=1000, flow_shift=FLOW_SHIFT,
    )
    # Move the small components ourselves; pipe.to("cuda") would try to move the
    # quantized encoder too (it is already on the GPU and must not be moved).
    pipe.transformer.to("cuda")
    pipe.vae.to("cuda")
    pipe.vae.enable_tiling()
    _free_vram()
    _PIPES[size] = pipe
    print(f"  {size} ready.")
    return pipe


## 6. Generate — all shots × all takes

Writes `takeN.mp4` into each shot folder. Already-generated takes are **skipped**, so re-running resumes where it left off. To force a re-do of a take, delete that mp4 in Drive.

(The loop runs any 1.3B shots first; if you opted a shot into 14B on a high-RAM runtime, it runs last.)

**Memory note:** prompts are encoded once up front, then the ~3GB text encoder is dropped from the GPU before rendering — the VAE decode of the longer shots needs that headroom. Each take also frees GPU memory when it finishes, and retries once if it hits an out-of-memory error.


In [ ]:
from diffusers.utils import export_to_video
import os, time, torch

# fast (1.3b) shots first, then slow (14b)
order = sorted(SHOTS.items(), key=lambda kv: 0 if kv[1]["model"] == "1.3b" else 1)

# 1) Encode every prompt once, then evict the ~3GB text encoder from the GPU.
#    The VAE decode of the 49/81-frame shots needs that headroom (a 49-frame
#    take OOMed at decode with the encoder still resident).
EMBEDS = {}
# inference_mode is essential: encode_prompt called outside the pipeline would
# otherwise build an autograd graph over the whole text encoder (GBs of VRAM).
with torch.inference_mode():
    for shot_id, cfg in order:
        pipe = get_pipe(cfg["model"])
        if shot_id not in EMBEDS:
            pe, ne = pipe.encode_prompt(
                prompt=cfg["prompt"],
                negative_prompt=NEGATIVE_PROMPT,
                do_classifier_free_guidance=True,
                num_videos_per_prompt=1,
                device="cuda",
            )
            EMBEDS[shot_id] = (pe.detach(), ne.detach())
for _p in _PIPES.values():
    _p.text_encoder = None
_free_vram()
print("Prompts encoded; text encoder evicted from GPU.")

# 2) Render. One OOM retry per take (after a memory sweep); a failing take is
#    skipped rather than killing the whole run.
for shot_id, cfg in order:
    pipe = _PIPES[cfg["model"]]
    shot_dir = os.path.join(OUTPUT_ROOT, shot_id)
    for take in range(1, cfg["takes"] + 1):
        dst = os.path.join(shot_dir, f"take{take}.mp4")
        if os.path.exists(dst):
            print(f"  {shot_id} take{take} already done, skipping.")
            continue
        seed = cfg["seeds"][(take - 1) % len(cfg["seeds"])]
        pe, ne = EMBEDS[shot_id]
        for attempt in (1, 2):
            gen_rng = torch.Generator(device="cuda").manual_seed(seed)
            t0 = time.time()
            print(f"  {shot_id} take{take} (model={cfg['model']}, seed={seed}, frames={cfg['num_frames']}, attempt {attempt}) ...")
            try:
                frames = pipe(
                    prompt_embeds=pe,
                    negative_prompt_embeds=ne,
                    height=HEIGHT, width=WIDTH,
                    num_frames=cfg["num_frames"],
                    num_inference_steps=cfg["steps"],
                    guidance_scale=cfg["guidance"],
                    generator=gen_rng,
                ).frames[0]
                export_to_video(frames, dst, fps=FPS)
                del frames
                _free_vram()
                print(f"    -> {dst}  ({time.time()-t0:.0f}s)")
                break
            except torch.cuda.OutOfMemoryError:
                _free_vram()
                if attempt == 1:
                    print("    OOM — memory swept, retrying this take once.")
                else:
                    print(f"    OOM twice — skipping {shot_id} take{take}; other takes still run.")
print("All requested shots/takes are on Drive ->", OUTPUT_ROOT)


## 7. Done — hand back to OpenMontage

Download the whole `vivaah/heroSection_v2/raw/` folder from Drive and place it at:
`D:\vivaah website preview\videos\heroSection_v2\raw\` (so you have `raw\shot1\take1.mp4`, etc.).

You do **not** need to pick the best take, trim, grade, or upscale — the OpenMontage compose stage does all of that (best-take pick, brand color grade, upscale to 1280×720, and the play-once-then-ping-pong-loop encode). Just make sure each shot has at least one clean take with **no watermark** and **no men in frame**.